# polars basics

[`polars`](https://pola.rs) is a fast DataFrame library — Rust's answer to
pandas. You'll use it to load, clean, and reshape tabular data before handing
arrays to a model. This builds on [ndarray basics](ndarray-basics.ipynb).

```{note}
In a notebook, `:dep` declarations and variables persist across cells in the
same kernel session — so we declare the crates and build the DataFrame once,
then reuse them below.
```

In [2]:
:dep polars = { version = "0.44", features = ["lazy", "ndarray"] }
use polars::prelude::*;

let df = df![
    "city"        => ["Paris", "Paris", "Berlin", "Berlin", "Rome"],
    "year"        => [2021, 2022, 2021, 2022, 2022],
    "temperature" => [12.1_f64, 12.8, 9.6, 10.2, 15.5],
]?;
println!("shape = {:?}", df.shape());
println!("{}", df);

shape = (5, 3)
shape: (5, 3)
┌────────┬──────┬─────────────┐
│ city   ┆ year ┆ temperature │
│ ---    ┆ ---  ┆ ---         │
│ str    ┆ i32  ┆ f64         │
╞════════╪══════╪═════════════╡
│ Paris  ┆ 2021 ┆ 12.1        │
│ Paris  ┆ 2022 ┆ 12.8        │
│ Berlin ┆ 2021 ┆ 9.6         │
│ Berlin ┆ 2022 ┆ 10.2        │
│ Rome   ┆ 2022 ┆ 15.5        │
└────────┴──────┴─────────────┘


## Selecting and filtering

The **lazy** API (`.lazy()` ... `.collect()`) lets you chain operations that
polars optimizes as a whole. Here we keep only 2022 rows and two columns:

In [3]:
let recent = df
    .clone()
    .lazy()
    .filter(col("year").eq(lit(2022)))
    .select([col("city"), col("temperature")])
    .collect()?;
println!("{}", recent);

shape: (3, 2)
┌────────┬─────────────┐
│ city   ┆ temperature │
│ ---    ┆ ---         │
│ str    ┆ f64         │
╞════════╪═════════════╡
│ Paris  ┆ 12.8        │
│ Berlin ┆ 10.2        │
│ Rome   ┆ 15.5        │
└────────┴─────────────┘


## Group-by aggregation

Average temperature per city — the DataFrame equivalent of a SQL `GROUP BY`:

In [4]:
let by_city = df
    .clone()
    .lazy()
    .group_by([col("city")])
    .agg([col("temperature").mean().alias("mean_temp")])
    .sort(["city"], Default::default())
    .collect()?;
println!("{}", by_city);

shape: (3, 2)
┌────────┬───────────┐
│ city   ┆ mean_temp │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ Berlin ┆ 9.9       │
│ Paris  ┆ 12.45     │
│ Rome   ┆ 15.5      │
└────────┴───────────┘


## From DataFrame to ndarray

When it's time to train a model, convert numeric columns to an `ndarray`
matrix. With the `ndarray` feature enabled (declared above), polars does this
directly — this is the bridge from data wrangling to the model chapters:

In [5]:
let numeric = df
    .clone()
    .lazy()
    .select([col("year"), col("temperature")])
    .collect()?;
// Wrapped in a block: the returned ndarray type isn't nameable across cells,
// so we keep it local (same pattern as fitted models elsewhere in the book).
{
    let matrix = numeric.to_ndarray::<Float64Type>(IndexOrder::C)?;
    println!("ndarray matrix ({} x {}):\n{}", matrix.nrows(), matrix.ncols(), matrix);
}

ndarray matrix (5 x 2):
[[2021, 12.1],
 [2022, 12.8],
 [2021, 9.6],
 [2022, 10.2],
 [2022, 15.5]]


()

You now have the two foundations — arrays and dataframes. Next: your first
model, [linear regression](../02-regression/linear-regression.ipynb).